Corrido sem masking e para pm.chi_sigma=True (caso C1).

In [1]:
%load_ext autoreload
%autoreload 2

%reload_ext autoreload

In [2]:
import sys
sys.path.insert(0, 'satellite_RFI_I')

In [3]:
##====================================================================================================================================##
## PARAMETERS AND IMPORTS
from satellite_RFI.src import tools
import importlib
import satellite_RFI.src.satellite_sims as ss_mod
importlib.reload(ss_mod)
print(ss_mod.__file__)
import time

from satellite_RFI.src.satellite_sims import satellite_sim as ss
from satellite_RFI.src import attenuation_function as af

import sys
sys.path.insert(0, '../../param_import/')
from imports import *
import param_multi_mask as pm
##====================================================================================================================================##
## OBSERVATION NAME + DATE
fname, dtime = tools.timepoint(fname=pm.file, date=None)

##====================================================================================================================================##
## FILE SAVE
## File path
file_loc = pm.data_save + pm.folder
## Filename
name = str(pm.file) + "_"


/users/bvitoria/.local/lib/python3.8/site-packages/satellite_RFI-0.1-py3.8.egg/satellite_RFI/src/satellite_sims.py
Data Path exists
Importing masking parameters...
Date of observation: 2019-02-25 02:40:11
Fname: 1551055211


In [4]:
##====================================================================================================================================##
## FREQUENCY

#Frequency range detection, covering possible missing values

if pm.fs_slice is not None and pm.fe_slice is not None:
    print("Frequeny range: " + str(pm.fs_slice) + "-" + str(pm.fe_slice) + " MHz")
elif pm.fs_slice is not None and pm.fe_slice is None:
    print("Frequeny range: " + str(pm.fs_slice) + " MHz to final value")
elif pm.fs_slice is None and pm.fe_slice is not None:
    print("Frequeny range: Intial value to " + str(pm.fe_slice) + " MHz")
elif pm.fs_slice is None and pm.fe_slice is None:
    print("Intial to Final frequency value")
else:
    print("Error, frequency value choices are incorrect")

freq_name = str(pm.fs_slice) + "-" + str(pm.fe_slice) + "_"

##====================================================================================================================================##
## MASK TYPE
#This part checks which RFI mask type(s) are applied - Masks are filters used to suppress or ignore certain types of data based on 
#(is not None indicates suppression of data, mask); Every combination of more than one mask also has its own elif block

## No mask
if (
    pm.mask_degree is None
    and pm.mask_temperature is None
    and pm.mask_temporal[0] is None
    and pm.mask_temporal[1] is None
    and pm.mask_pixel_timeline is None
):
    mask_name = "no-mask_"
    print(pm.mask_type)
    
    
## Angular mask
elif (
    pm.mask_degree is not None
    and pm.mask_temperature is None
    and pm.mask_temporal[0] is None
    and pm.mask_temporal[1] is None
    and pm.mask_pixel_timeline is None
):
    mask_name = "degree-" + pm.mask_degree + "_"
    print(pm.mask_type)
    
    
## Temperature mask
elif (
    pm.mask_temperature is not None
    and pm.mask_degree is None
    and pm.mask_temporal[0] is None
    and pm.mask_temporal[1] is None
    and pm.mask_pixel_timeline is None
):
    mask_name = "thermal-" + str(pm.mask_temperature) + "_"
    print(pm.mask_type)
    
    
## Temporal mask
elif (
    pm.mask_degree is None
    and pm.mask_temperature is None
    and (pm.mask_temporal[0] is not None or pm.mask_temporal[1] is not None)
    and pm.mask_pixel_timeline is None
):
    mask_name = "temporal_"
    print(pm.mask_type)
    
    
## Timeline Pixel mask
elif (
    pm.mask_degree is None
    and pm.mask_temperature is None
    and pm.mask_temporal[0] is None
    and pm.mask_temporal[1] is None
    and pm.mask_pixel_timeline is not None
):
    mask_name = "pix_timeline_"+str(pm.mask_pixel_timeline) + "_"
    print(pm.mask_type)

    
## Angular+Thermal mask
elif (
    pm.mask_degree is not None
    and pm.mask_temperature is not None
    and pm.mask_temporal[0] is None
    and pm.mask_temporal[1] is None
    and pm.mask_pixel_timeline is None
):
    mask_name = (
        "degree-" + pm.mask_degree + "_thermal-" + str(pm.mask_temperature) + "_"
    )
    print(pm.mask_type)
    
    
## Angular+Temporal mask
elif (
    pm.mask_degree is not None
    and pm.mask_temperature is None
    and (pm.mask_temporal[0] is not None or pm.mask_temporal[1] is not None)
    and pm.mask_pixel_timeline is None
):
    mask_name = "degree-" + pm.mask_degree + "_temporal_"
    print(pm.mask_type)
    
    
## Thermal+Temporal mask
elif (
    pm.mask_degree is None
    and pm.mask_temperature is not None
    and (pm.mask_temporal[0] is not None or pm.mask_temporal[1] is not None)
    and pm.mask_pixel_timeline is None
):
    mask_name = "thermal-" + str(pm.mask_temperature) + "_temporal_"
    print(pm.mask_type)
    
    
## Angular+Thermal+Temporal
elif (
    pm.mask_degree is not None
    and pm.mask_temperature is not None
    and pm.mask_temporal[0] is not None
    and pm.mask_temporal[1] is not None
    and pm.mask_pixel_timeline is None
):
    mask_name = (
        "degree-"
        + pm.mask_degree
        + "_thermal-"
        + str(pm.mask_temperature)
        + "_temporal_"
    )
    print(pm.mask_type)
else:
    print(pm.mask_type)

    
## Temporal mask
#if pm.ts_slice is not None and pm.te_slice is not None: 
#To determine the time range of data that should be analyzed, and label it accordingly for use in filenames

#pm.ts_slice → user-defined start time in seconds
#pm.te_slice → user-defined end time in seconds
#pm.nd_s0 → array or list of all time values in the dataset

if pm.ts_slice is not None and pm.te_slice is not None:
    print(
        "Time range: "
        + str(pm.ts_slice)
        + " seconds to "
        + str(pm.te_slice)
        + " seconds."
    )
    time_name = str(pm.ts_slice) + "-" + str(pm.te_slice) + "_"
elif pm.ts_slice is not None and pm.te_slice is None:
    print("Time range: " + str(pm.ts_slice) + " seconds to final time.")
    time_name = str(pm.ts_slice) + "-" + str(np.round(pm.nd_s0[-1], 2)) + "_"
elif pm.ts_slice is None and pm.te_slice is not None:
    print("Time range: initial time to " + str(pm.te_slice) + " seconds.")
    time_name = str(np.round(pm.nd_s0[0], 2)) + "-" + str(pm.te_slice) + "_"
elif pm.ts_slice is None and pm.te_slice is None:
    print("Time range: inital to final time.")
    time_name = (
        str(np.round(pm.nd_s0[0], 2)) + "-" + str(np.round(pm.nd_s0[-1], 2)) + "_"
    )
else:
    print("Error, time range choices are incorrect.")

## Time average
if pm.time_average is not None:
    time_average_name = "time_average_" + str(pm.time_average) + "_"
else:
    time_average_name = ""

Frequeny range: 1100-1350 MHz
Masking: None.
Time range: inital to final time.


In [5]:
##====================================================================================================================================##
## DAMPNER
#"goob" dampening — a Gaussian function that suppresses spectral energy outside a defined band
#If no sigma value is given, the system will randomly assign a Gaussian width
#Gaussian dampening is used in signal processing to smooth transitions and reduce spectral leakage

if pm.dampner is None:
    print("No dampening.")
    dampner_name = "no-dampening_"
elif pm.dampner == "goob":
    print("Gaussian Out of Band dampening given.")
    if pm.dampner_sigma is None:
        print("Random dampner values given.")
        dampner_name = pm.dampner + "-random_"
    else:
        print("Damper values fixed to " + str(pm.dampner_sigma) + " level.")
        dampner_name = pm.dampner + "-" + str(pm.dampner_sigma) + "_"

No dampening.


In [6]:
#====================================================================================================================================##
## CHI SQUARE SIGMA

#configuring how the Chi-Square denominator is set when analyzing residuals (after removing RFI or applying masks)
#radiometer equation gives the expected noise level (standard deviation) in a radio signal measurement
#It's used to quantify how much random noise you should expect based on your system parameters
#pm.chi_sigma -> a Boolean that determines whether to use a proper noise model or a fixed value in the denominator

if pm.chi_sigma == True:
    print("The Chi-Sqaure denominator will be: radiometer equation.")
    chi_sig_name = "fraction_"
elif pm.chi_sigma == False:
    print("The Chi-Sqaure denominator will be: 1.")
    chi_sig_name = "residual_"
else:
    print("Error in selecting the Chi-Square sigma option.")


The Chi-Sqaure denominator will be: radiometer equation.


In [7]:
#====================================================================================================================================##
file_save = (
    file_loc
    + name
    + freq_name
    + time_name
    + mask_name
    + dampner_name
    + chi_sig_name
    + time_average_name
    + pm.save_suffix
)
print("File location is: " + file_save)

File location is: /idia/projects/hi_im/satellite_rfi/Testing/1551055211/sat_12/1551055211_1100-1350_774.75-6474.34_no-mask_no-dampening_fraction_iara


In [8]:
##====================================================================================================================================##
## INITIALIZING THE FUNCTION

#using the class satellite_sim from file satellite_sims.py
sat = ss(
    file_name=fname,
    sats_only=False,
    data_loc=pm.data_save,
    sat_loc=pm.data_save,
    survey_info=[pm.nd_s0, pm.nd_s0_coords, pm.frequency],
    sat_info=pm.satellite_catalogue,
    plots_loc=pm.data_plot,
    sat_beam=pm.beam_model + "_beam_" + str(pm.fs) + "_" + str(pm.fe) + "MHz",
    frequency_range=[pm.fs, pm.fe],
    constellations=pm.constellations_remain,
    nearby_satellites=pm.nearby_constellations,
    verbose=False,
)


In [9]:
## INTIAL ALPHA DICTIONARY VALUES
## NOTE - There can be a length issue, the length should be the same as number of signals available
alpha_dic = np.zeros(pm.no_signals)

## INTIAL SIGMA DICTIONARY VALUES
if pm.dampner is not None:
    # Note- There can be a length issue here, be aware
    if pm.dampner_sigma is not None:
        print("Dampner will be fixed.")
        sigma_dic = pm.dampner_sigma * np.ones(21)
        # The input values for the Optimization of the Chi-Sqaure fitting
        input_param = alpha_dic
    else:
        print("Dampner will be random.")
        sigma_dic = np.ones(21)
        input_param = np.concatenate((alpha_dic, sigma_dic))

else:
    sigma_dic = None
    input_param = alpha_dic


In [10]:
print(sat.sats_only)

False


In [11]:
#====================================================================================================================================##
## EXCECUTING THE INITIAL FUNCTION
try:
    sat.excecute(
        a_param=alpha_dic,
        obs_time_start=pm.ts_slice,
        obs_time_end=pm.te_slice,
        obs_frequency_start=pm.fs_slice,
        obs_frequency_end=pm.fe_slice,
        file_bias_choice=pm.bias,
        add_sub=[True, False],
        attenuation_func=pm.dampner,
        attenuation_sigma=sigma_dic,
        bandsize=None,
        verbose=True,
    )
    print("✅ excecute() completed successfully")
except Exception:
    print("❌ excecute() raised an exception:")
    traceback.print_exc()


Number of signals inside choice frequency range:  21
✅ excecute() completed successfully


In [12]:
##TRYING TO GENERATE THE OBSERVATIONAL DATA ONLY ONE TIME

def generate_masked_data():
    data_raw = sat.calibration_data_slice.T #

    if (
        pm.mask_degree is None
        and pm.mask_temperature is None
        and pm.mask_temporal[0] is None
        and pm.mask_temporal[1] is None
        and pm.mask_pixel_timeline is None
    ):
        mask = None

    elif (
        pm.mask_degree is not None
        and pm.mask_temperature is None
        and pm.mask_temporal[0] is None
        and pm.mask_temporal[1] is None
        and pm.mask_pixel_timeline is None
    ):
        mask = sat.mask_nearby_satellites_slice

    elif (
        pm.mask_temperature is not None
        and pm.mask_degree is None
        and pm.mask_temporal[0] is None
        and pm.mask_temporal[1] is None
        and pm.mask_pixel_timeline is None
    ):
        max_k = pm.mask_temperature
        zero_arr = np.zeros(data_raw.T.shape)
        zero_arr[np.where(data_raw.T > max_k)] = 1
        mask = zero_arr.T

    elif (
        pm.mask_pixel_timeline is not None
        and all(x is None for x in [pm.mask_degree, pm.mask_temperature, pm.mask_temporal[0], pm.mask_temporal[1]])
    ):
        mask = tools.time_line_masker(data_in=data_raw, t_value=pm.mask_pixel_timeline)

    elif (
        pm.mask_degree is not None
        and pm.mask_temperature is not None
        and pm.mask_temporal[0] is None
        and pm.mask_temporal[1] is None
    ):
        max_k = pm.mask_temperature
        temp_mask = np.zeros(data_raw.T.shape)
        temp_mask[np.where(data_raw.T > max_k)] = 1
        mask = np.logical_or(temp_mask.T, sat.mask_nearby_satellites_slice)

    elif (
        pm.mask_degree is not None
        and pm.mask_temperature is not None
        and pm.mask_temporal[0] is not None
        and pm.mask_temporal[1] is not None
    ):
        max_k = pm.mask_temperature
        temp_mask = np.zeros(data_raw.T.shape)
        temp_mask[np.where(data_raw.T > max_k)] = 1
        mask = np.logical_or(temp_mask.T, sat.mask_nearby_satellites_slice)

    else:
        raise ValueError("Invalid masking configuration.")

    return np.ma.array(data=data_raw, mask=mask)

In [13]:
obs_data = generate_masked_data()

In [14]:
##====================================================================================================================================##

#CHISQ now only generates the simulation within it, and intakes the fixed observational data
def chisq_func2(params, obs_data):
    a_param = params[:pm.no_signals]

    if pm.dampner is not None:
        if pm.dampner_sigma is None:
            s_param = params[21:]
        else:
            s_param = pm.dampner_sigma * np.ones(21)
    else:
        s_param = None

    sat.excecute(
        a_param=a_param,
        obs_time_start=pm.ts_slice,
        obs_time_end=pm.te_slice,
        obs_frequency_start=pm.fs_slice,
        obs_frequency_end=pm.fe_slice,
        file_bias_choice=pm.bias,
        add_sub=[True, False],
        attenuation_func=pm.dampner,
        attenuation_sigma=s_param,
        bandsize=None,
        verbose=False,
    )

    simulation = np.ma.array(data=sat.simulation_TOD_slice.T, mask=obs_data.mask)

    #Chi-square computation
    sig = obs_data if pm.chi_sigma else 1
    chi_sq = np.ma.sum((simulation-obs_data)**2 / sig**2)

    print(chi_sq)
    return chi_sq


In [15]:
chi_sq_value = chisq_func2(params=alpha_dic, obs_data=obs_data) #just testing the function to see if everything is ok

117746.14337149158


In [16]:
##====================================================================================================================================##
## PRIOR-BOUNDARY INFORMATION
## Boundary values
bnd_val = (0.0, None)
print("Boundary values are set at: ", bnd_val)
## Seeting bounds for the Chi-Square
if pm.dampner is not None:
    if pm.dampner_sigma is None:
        bnds = [bnd_val for bnd_i in range(2 * sat.alpha_len)]
    elif pm.dampner_sigma is not None:
        bnds = [bnd_val for bnd_i in range(sat.alpha_len)]
    else:
        print("Error in setting in boundary conditons.")
else:
    bnds = [bnd_val for bnd_i in range(sat.alpha_len)]


Boundary values are set at:  (0.0, None)


In [17]:
##====================================================================================================================================##
## RUNNING THE OPTIMIZATION PARAMETERS
print("Running optimization")
start = time.perf_counter()
signal_PL = opt.minimize(
    fun = chisq_func2,
    x0 = input_param,  # Set to none becuase the number of signals will be determined by the bandsize
    args=(obs_data,), #need to put this extra argument now that the function expects the observatonal data
    method = "Powell",
    bounds = bnds,
    tol = 1e-6,
    options={"maxiter": 20},
)
elapsed = time.perf_counter() - start
print(f"This took {elapsed} seconds, or {elapsed/(60*60)} hours".format())


Running optimization
117746.14337149158
116774.87631436801
115910.58971020093
115118.55441264305
114916.87541833468
115789.15110475844
114875.38190723583
114850.16132558053
114846.97445880274
114846.42244161063
114846.29449992947
114846.29438933688
114846.29436159862
114846.29436159845
114846.2943616097
114580.47238138535
116712.58875618571
114449.31712391392
114451.20194672479
114445.17534584129
114445.17689054129
114445.17534444839
114445.17534402775
114445.17534402802
114445.17534402807
115566.87189573991
133455.52470461084
113404.36716106115
113585.65641827171
113268.55535193565
113371.29503015777
113249.90358672311
113249.83103000975
113249.82187046277
113249.8218573016
113249.82185729772
113249.82185729958
108426.83268187278
270320.0591388188
93600.82285177786
94896.1460087557
93550.69382152408
93524.49994067484
93524.26713875277
93524.26521142205
93524.26520607084
93524.26520608699
93524.26520609816
677610.1148256084
2828129.560543848
275295.1182626537
258246.07162956358
149479.

In [18]:
##====================================================================================================================================##
## BEST FIT VALUES
if pm.dampner is not None:
    # Note- There can be a length issue here, be aware
    if pm.dampner_sigma is not None:
        print("Dampner will be fixed.")
        sigma_param_bf = pm.dampner_sigma * np.ones(21)
        # The input values for the Optimization of the Chi-Sqaure fitting
        a_param_bf = signal_PL.x
    else:
        print("Dampner was best fitted.")
        sigma_param_bf = signal_PL.x[21:]
        a_param_bf = signal_PL.x[:21]

else:
    sigma_param_bf = None
    a_param_bf = signal_PL.x

In [19]:
##====================================================================================================================================##
## RUNNING SECOND INITIALIZATION
print("Running 2nd init")
sat2 = ss(
    file_name=fname,
    sats_only=False,
    data_loc=pm.data_save,
    sat_loc=pm.data_save,
    survey_info=[pm.nd_s0, pm.nd_s0_coords, pm.frequency],
    sat_info=pm.satellite_catalogue,
    plots_loc=pm.data_plot,
    sat_beam=pm.beam_model + "_beam_" + str(pm.fs) + "_" + str(pm.fe) + "MHz",
    frequency_range=[pm.fs, pm.fe],
    constellations=pm.constellations_remain,
    nearby_satellites=pm.nearby_constellations,
    verbose=False,
)

sat2.excecute(
    a_param=a_param_bf,
    obs_time_start=pm.ts_slice,
    obs_time_end=pm.te_slice,
    obs_frequency_start=pm.fs_slice,
    obs_frequency_end=pm.fe_slice,
    file_bias_choice=pm.bias,
    add_sub=[True, False],
    attenuation_func=pm.dampner,
    attenuation_sigma=sigma_param_bf,
    bandsize=None,
    verbose=False,
)


Running 2nd init


In [20]:
##====================================================================================================================================##
## SAVING DATA INFORMATION
data_info = {
    "initial" : input_param,
    "time" : [pm.ts_slice, pm.te_slice],
    "frequency_slice" : [pm.fs_slice, pm.fe_slice],
    "best-fit" : signal_PL.x,
    "chi2_value" : signal_PL.fun,
    "chi2_div" : signal_PL.fun / sat2.simulation_TOD_slice.size,
}

file_save = "/users/bvitoria/workspace/dados_iara"
pickle.dump(data_info, open(file_save + ".p", "wb"))
